explain how we downloaded the fairface dataset using kaggle. use `cmd + /` on macbook to uncomment and download the data for yourself

also make sure to write the readme explaining setup instructions. notably, venv and kernel setup.

`python -m ipykernel install --user --name=facial-bias --display-name "Python (facial-bias)"`

In [1]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("aibloy/fairface")

# print("Path to dataset files:", path)

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv('../FairFace/train_labels.csv')

races = df['race'].unique()
print(races)


<StringArray>
[     'East Asian',          'Indian',           'Black',           'White',
  'Middle Eastern', 'Latino_Hispanic', 'Southeast Asian']
Length: 7, dtype: str


make datasets

In [3]:
# testing dataset: 40 per race
test_parts = []
remaining_df = df.copy()

for race in races:
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=40, random_state=42)
    test_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

test_df = pd.concat(test_parts)

# dataset A: Balanced (150 per race)
A_parts = []

for race in races:
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=150, random_state=42)
    A_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

A_df = pd.concat(A_parts)

# dataset B: Moderately Imbalanced
dominant_race = 'White'
B_parts = []

# dominant: 450
dom_df = remaining_df[remaining_df['race'] == dominant_race]
dom_sample = dom_df.sample(n=450, random_state=42)
B_parts.append(dom_sample)
remaining_df = remaining_df.drop(dom_sample.index)

# others: ~100 each
for race in races:
    if race == dominant_race:
        continue
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=100, random_state=42)
    B_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

B_df = pd.concat(B_parts)

# dataset C: Highly Imbalanced
C_parts = []

# dominant: 700
dom_df = remaining_df[remaining_df['race'] == dominant_race]
dom_sample = dom_df.sample(n=700, random_state=42)
C_parts.append(dom_sample)
remaining_df = remaining_df.drop(dom_sample.index)

# others: ~50 each
for race in races:
    if race == dominant_race:
        continue
    race_df = remaining_df[remaining_df['race'] == race]
    sample = race_df.sample(n=50, random_state=42)
    C_parts.append(sample)
    remaining_df = remaining_df.drop(sample.index)

C_df = pd.concat(C_parts)


print("Test:", "size:", len(test_df), test_df['race'].value_counts())
print("A:", "size:", len(A_df), A_df['race'].value_counts())
print("B:", "size:", len(B_df), B_df['race'].value_counts())
print("C:", "size:", len(C_df), C_df['race'].value_counts())

Test: size: 280 race
East Asian         40
Indian             40
Black              40
White              40
Middle Eastern     40
Latino_Hispanic    40
Southeast Asian    40
Name: count, dtype: int64
A: size: 1050 race
East Asian         150
Indian             150
Black              150
White              150
Middle Eastern     150
Latino_Hispanic    150
Southeast Asian    150
Name: count, dtype: int64
B: size: 1050 race
White              450
East Asian         100
Indian             100
Black              100
Middle Eastern     100
Latino_Hispanic    100
Southeast Asian    100
Name: count, dtype: int64
C: size: 1000 race
White              700
East Asian          50
Indian              50
Black               50
Middle Eastern      50
Latino_Hispanic     50
Southeast Asian     50
Name: count, dtype: int64


save datasets to csvs

In [ ]:
test_df.to_csv('../FairFace/test.csv', index=False)
A_df.to_csv('../FairFace/train_A.csv', index=False)
B_df.to_csv('../FairFace/train_B.csv', index=False)
C_df.to_csv('../FairFace/train_C.csv', index=False)

convert these images to embeddings using DeepFace pretrained model

In [6]:
from deepface import DeepFace
import pandas as pd
import numpy as np
import os

IMG_ROOT = "../FairFace"  # correct root

def make_embeddings(csv_in, csv_out):
    df = pd.read_csv(csv_in)
    embeddings = []

    for path in df["file"]:
        full_path = os.path.join(IMG_ROOT, path)

        if not os.path.exists(full_path):
            print("Missing:", full_path)
            embeddings.append([np.nan]*512)  # skip safely
            continue

        emb = DeepFace.represent(
            img_path=full_path,
            model_name="Facenet",
            enforce_detection=False,
            detector_backend="skip"  # faster + avoids face detection issues
        )[0]["embedding"]

        embeddings.append(emb)

    emb_df = pd.DataFrame(embeddings)
    out = pd.concat([emb_df, df.reset_index(drop=True)], axis=1)
    out.to_csv(csv_out, index=False)

# run
make_embeddings("../FairFace/train_A.csv", "../datasets/train_A_emb.csv")
make_embeddings("../FairFace/train_B.csv", "../datasets/train_B_emb.csv")
make_embeddings("../FairFace/train_C.csv", "../datasets/train_C_emb.csv")
make_embeddings("../FairFace/test.csv",   "../datasets/test_emb.csv")